In [1]:
#Imports
import pandas as pd
import numpy as np
from scipy.stats import f,t
from google.colab import files
from sklearn.model_selection import train_test_split, KFold

In [2]:
# Add Dataset
uploaded = files.upload()
df_housing = pd.read_csv('Revised - Merged Housing Dataset (by ZipCode).csv')
df_stab = pd.read_csv('final_zip_stabilization_dataset.csv')
# df_pluto_stab = pd.read_csv('pluto_with_rent_stabilization_flag.csv')
#Data Setup


Saving final_zip_stabilization_dataset.csv to final_zip_stabilization_dataset.csv
Saving Revised - Merged Housing Dataset (by ZipCode).csv to Revised - Merged Housing Dataset (by ZipCode).csv


In [3]:
# Multiple Linear Regression Function

def mlr(Xk, Y):
  n = len(Y) #Number of Samples
  p = Xk.shape[1] #Number of Predictors,
  X = np.column_stack((np.ones(n),Xk)) # adds the intercept

  betahat = np.linalg.solve(X.T@X, X.T@Y)
  yhat = X @ betahat
  resid = Y - yhat

  #Sum of Squaresns
  sse = np.sum(resid**2)
  sst = np.var(Y)*(n-1)
  ssm = sst-sse

  #Mean Squares
  mse = sse/(n-p-1)
  mst = sst/(n-1)
  msm = ssm/p
  rmse = np.sqrt(mse)
  fstats = msm/mse #shows how useful each variable is at predicting


  # Variance-Covariance of Coefficients
  var_betahat = mse * np.linalg.inv(X.T @ X)
  se_betahat = np.sqrt(np.diagonal(var_betahat))

  # T-statistics for each variable
  t_stats = betahat / se_betahat

  pval = 2 * (1 - t.cdf(np.abs(t_stats), n - p - 1))

  #Model Fit
  r2 = 1 - (sse/sst) #ranges from [0,1]
  r2_adjusted = 1 - (mse/mst) #Incase of poor model fit

  # Akaike Information Criterion
  aic = n * np.log(sse / n) + 2 * (p + 1) + n * (1 + np.log(2 * np.pi))

  return r2, r2_adjusted, aic, betahat, se_betahat, pval


In [4]:
# DATA PREP
df_housing['zip'] = pd.to_numeric(df_housing['zip'], errors='coerce')
df_housing = df_housing.dropna(subset=['zip'])
df_housing['zip'] = df_housing['zip'].astype(int)

#Merges data on shared components
df_merged = pd.merge(df_housing, df_stab, left_on='zip',right_on ='zipcode', how='inner')

# FEATURE ENGINEERING
# Calculate Housing Supply with Vacancy Rate
df_merged['vacancy_rate'] = df_merged['vacant_units'] / df_merged['housing_units_total']

# Homeownership Rate
df_merged['homeownership_rate'] = df_merged['owner_occupied_units'] / df_merged['housing_units_total']

# Construction Share (Everything built 2010 or later)
df_merged['new_construction_share'] = (df_merged['built_2020_or_later'] + df_merged['built_2010_2019']) / df_merged['housing_units_total']

# Renter Density (Share of units occupied by renters)
df_merged['renter_density'] = df_merged['renter_occupied_units'] / df_merged['housing_units_total']

# Pre-War Building Share (Built 1939 or earlier)
df_merged['pre_war_share'] = df_merged['built_1939_or_earlier'] / df_merged['housing_units_total']

df_merged

,Unnamed: 0,geoid,zip,borough,rent_burden_total_renter_households,rent_lt_10pct_income,rent_10_to_14_9pct_income,rent_15_to_19_9pct_income,rent_20_to_24_9pct_income,rent_25_to_29_9pct_income,...,total_units,stabilized_units,stabilized_buildings,total_buildings,stabilization_share,vacancy_rate,homeownership_rate,new_construction_share,renter_density,pre_war_share
0,31,86000US10001,10001,Manhattan,11939.0,917.0,1502.0,1801.0,1484.0,1181.0,...,22303.0,12310.0,127,356,0.551944,0.142678,0.206908,0.290314,0.650414,0.270375
1,32,86000US10002,10002,Manhattan,29732.0,1688.0,2438.0,2857.0,3466.0,3197.0,...,36840.0,14431.0,601,1180,0.391721,0.101730,0.167682,0.057573,0.730588,0.389424
2,33,86000US10003,10003,Manhattan,15410.0,1031.0,2255.0,1722.0,1455.0,1355.0,...,33576.0,18652.0,602,1479,0.555516,0.194325,0.298984,0.024102,0.506691,0.543386
3,34,86000US10004,10004,Manhattan,1060.0,21.0,119.0,255.0,84.0,39.0,...,4492.0,1248.0,5,37,0.277827,0.170004,0.358255,0.098353,0.471740,0.615932
4,35,86000US10005,10005,Manhattan,4323.0,77.0,515.0,461.0,664.0,559.0,...,6871.0,2923.0,8,25,0.425411,0.205571,0.140024,0.024372,0.654405,0.516652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,253,86000US11691,11691,Queens,16665.0,1049.0,1126.0,915.0,1993.0,1893.0,...,24882.0,9036.0,156,5608,0.363154,0.042913,0.264789,0.049892,0.692298,0.183325
174,254,86000US11692,11692,Queens,5400.0,342.0,471.0,405.0,408.0,251.0,...,8348.0,83.0,16,2083,0.009943,0.040998,0.317291,0.155674,0.641711,0.116934
175,255,86000US11693,11693,Queens,2847.0,112.0,238.0,240.0,301.0,375.0,...,4727.0,340.0,48,1526,0.071927,0.061297,0.457953,0.063661,0.480750,0.188619
176,256,86000US11694,11694,Queens,3963.0,310.0,263.0,685.0,572.0,284.0,...,9553.0,1423.0,29,3513,0.148958,0.100573,0.462877,0.033488,0.436550,0.316369


## Model Run #1

In [5]:
# VARIABLE SELECTION
# Y is our target(Median Gross Rent), X represents variables we think affect rent
cols_to_keep = ['median_gross_rent', 'stabilization_share', 'median_household_income', 'vacancy_rate', 'total_population']
df_model = df_merged[cols_to_keep].replace([np.inf, -np.inf], np.nan).dropna()

Y = df_model['median_gross_rent'].values
Xk = df_model[['stabilization_share', 'median_household_income', 'vacancy_rate', 'total_population']]
feature_names = ['Intercept', 'Stabilization Share', 'Median Income', 'Vacancy Rate', 'Total Population']

# Multicollinearity Check
# Calculate the Correlation Matrix of the predictors
corr_matrix = np.corrcoef(Xk.values, rowvar=False)

# Calculate Variance Inflation Factors
inverse_corr_matrix = np.linalg.inv(corr_matrix)
vifs = np.diagonal(inverse_corr_matrix)


print("======================================================================")
print(f"MULTICOLLINEARITY DIAGNOSTICS (VIF)")
print("======================================================================")
print(f"{'Predictor Variable':<25} | {'VIF Score':<10} | {'Status':<15}")
print("-" * 65)

for i in range(len(vifs)):
    vif_val = vifs[i]
    if vif_val > 5:
        status = "DANGER (>5)"
    elif vif_val > 2.5:
        status = "MODERATE"
    else:
        status = "SAFE"
    print(f"{feature_names[i]:<25} | {vif_val:<10.4f} | {status:<15}")

MULTICOLLINEARITY DIAGNOSTICS (VIF)
Predictor Variable        | VIF Score  | Status         
-----------------------------------------------------------------
Intercept                 | 1.0147     | SAFE           
Stabilization Share       | 1.6463     | SAFE           
Median Income             | 1.4091     | SAFE           
Vacancy Rate              | 1.2334     | SAFE           


In [6]:
# MODEL TRAINING

X_train, X_test, Y_train, Y_test = train_test_split(Xk, Y, test_size=0.20, random_state=42)
r_squared_train, r2_adj_train, aic_train, betahat, se_betahat, p_values_array = mlr(X_train, Y_train)

betahat = np.atleast_1d(betahat)
p_values_array = np.atleast_1d(p_values_array)

# MODEL TESTING
n_test = len(Y_test)
X_test_intercept = np.column_stack((np.ones(n_test), X_test))

#Predictions based on test
Y_predictions = X_test_intercept @ betahat

# Calculate Accuracy
residuals = Y_test - Y_predictions
test_sse = np.sum(residuals**2)
test_sst = np.sum((Y_test - np.mean(Y_test))**2)

test_r_squared = 1 - (test_sse / test_sst)
#Average of what we were off by (in dollars)
test_rmse = np.sqrt(test_sse / n_test)

print("======================================================================")
print(f"MACHINE LEARNING EVALUATION: TRAIN VS. TEST")
print("======================================================================")
print(f"Data Split:           {len(Y_train)} Training Neighborhoods | {n_test} Testing Neighborhoods")
print(f"Training R-Squared:   {r_squared_train:.4f} (Accuracy on data it studied)")
print(f"Testing R-Squared:    {test_r_squared:.4f} (Accuracy on HIDDEN data)")
print(f"Test RMSE (Error):   ${test_rmse:.2f} (Average prediction error in dollars)")

print("\n======================================================================")
print(f"MODEL FIT SUMMARY (BASED ON TRAINING SET)")
print("======================================================================")
print(f"AIC Score:          {aic_train:.4f}")
print("======================================================================")
print(f"{'Predictor Variable':<22} | {'Coefficient':<12} | {'P-Value':<10} | {'Significant?':<12}")
print("-" * 65)

for i in range(len(feature_names)):
    sig = "YES" if p_values_array[i] < 0.05 else "NO"
    coef_str = f"{betahat[i]:.6f}"
    pval_str = f"{p_values_array[i]:.4f}"

    print(f"{feature_names[i]:<22} | {coef_str:<12} | {pval_str:<10} | {sig:<12}")
print("======================================================================")

MACHINE LEARNING EVALUATION: TRAIN VS. TEST
Data Split:           140 Training Neighborhoods | 35 Testing Neighborhoods
Training R-Squared:   0.7819 (Accuracy on data it studied)
Testing R-Squared:    0.8664 (Accuracy on HIDDEN data)
Test RMSE (Error):   $248.36 (Average prediction error in dollars)

MODEL FIT SUMMARY (BASED ON TRAINING SET)
AIC Score:          1991.3251
Predictor Variable     | Coefficient  | P-Value    | Significant?
-----------------------------------------------------------------
Intercept              | 739.377977   | 0.0000     | YES         
Stabilization Share    | 140.572175   | 0.1689     | NO          
Median Income          | 0.011379     | 0.0000     | YES         
Vacancy Rate           | 1769.214451  | 0.0004     | YES         
Total Population       | 0.000535     | 0.6148     | NO          


Cross Validation Test for Model #1

In [7]:
# ======================================================================
# CROSS-VALIDATION & FINAL MODEL TEST
# ======================================================================
# We split on df_model which you already cleaned in your block above
df_train, df_test = train_test_split(df_model, test_size=0.20, random_state=42)

Y_train = df_train['median_gross_rent'].values
Y_test = df_test['median_gross_rent'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n======================================================================")
print("Evaluating Model (Stabilization, Income, Vacancy, Population)...")
print("======================================================================")

model_features = ['stabilization_share', 'median_household_income', 'vacancy_rate', 'total_population']

X_train_model = df_train[model_features].values
X_test_model = df_test[model_features].values

# 1. Run Cross-Validation on the Training Data
cv_r2_scores = []
cv_rmse_scores = []
fold_number = 1

for train_index, val_index in kf.split(X_train_model):
    X_fold_train, X_fold_val = X_train_model[train_index], X_train_model[val_index]
    Y_fold_train, Y_fold_val = Y_train[train_index], Y_train[val_index]

    # Train on 4 folds
    _, _, _, betahat_fold, _, _ = mlr(X_fold_train, Y_fold_train)

    # Validate on the 1 remaining fold
    n_val = len(Y_fold_val)
    X_fold_val_int = np.column_stack((np.ones(n_val), X_fold_val))
    fold_predictions = X_fold_val_int @ betahat_fold

    # Calculate Fold Metrics
    fold_sse = np.sum((Y_fold_val - fold_predictions)**2)
    fold_sst = np.sum((Y_fold_val - np.mean(Y_fold_val))**2)

    fold_r2 = 1 - (fold_sse / fold_sst)
    fold_rmse = np.sqrt(fold_sse / n_val)

    cv_r2_scores.append(fold_r2)
    cv_rmse_scores.append(fold_rmse)

    print(f"Fold {fold_number}: R-Squared = {fold_r2:.4f} | RMSE = ${fold_rmse:.2f}")
    fold_number += 1

# --- STANDARD DEVIATION ---
cv_r2_mean = np.mean(cv_r2_scores)
cv_r2_std = np.std(cv_r2_scores)

cv_rmse_mean = np.mean(cv_rmse_scores)
cv_rmse_std = np.std(cv_rmse_scores)

print("----------------------------------------------------------------------")
print(f"-> Cross-Validation R-Squared: {cv_r2_mean:.4f} (± {cv_r2_std:.4f})")
print(f"-> Cross-Validation RMSE:      ${cv_rmse_mean:.2f} (± ${cv_rmse_std:.2f})")
print("======================================================================")

# 2. Final Test!
_, _, _, betahat_final, _, _ = mlr(X_train_model, Y_train)

X_test_model_int = np.column_stack((np.ones(len(Y_test)), X_test_model))

final_predictions = X_test_model_int @ betahat_final
final_sse = np.sum((Y_test - final_predictions)**2)
final_sst = np.sum((Y_test - np.mean(Y_test))**2)
final_test_r2 = 1 - (final_sse / final_sst)
final_test_rmse = np.sqrt(final_sse / len(Y_test))

print(f"-> FINAL Test Set R-Squared:           {final_test_r2:.4f}")
print(f"-> FINAL Test Set RMSE:               ${final_test_rmse:.2f}\n")


Evaluating Model (Stabilization, Income, Vacancy, Population)...
Fold 1: R-Squared = 0.6869 | RMSE = $233.30
Fold 2: R-Squared = 0.6509 | RMSE = $309.03
Fold 3: R-Squared = 0.8002 | RMSE = $289.15
Fold 4: R-Squared = 0.6315 | RMSE = $417.01
Fold 5: R-Squared = 0.8945 | RMSE = $226.58
----------------------------------------------------------------------
-> Cross-Validation R-Squared: 0.7328 (± 0.0998)
-> Cross-Validation RMSE:      $295.01 (± $68.71)
-> FINAL Test Set R-Squared:           0.8664
-> FINAL Test Set RMSE:               $248.36



## Model Run #2

In [8]:
# VARIABLE SELECTION
cols_to_keep = ['median_gross_rent', 'median_household_income', 'vacancy_rate', 'homeownership_rate', 'new_construction_share']
df_model = df_merged[cols_to_keep].replace([np.inf, -np.inf], np.nan).dropna()

Y = df_model['median_gross_rent'].values
Xk = df_model[['median_household_income', 'vacancy_rate', 'homeownership_rate', 'new_construction_share']]

feature_names = ['Intercept', 'Median Income', 'Vacancy Rate', 'Homeownership Rate', 'New Construction Share']

# Multicollinearity Check
# --- NEW MATH: MULTICOLLINEARITY CHECKS ---
# 1. Calculate the Correlation Matrix of the predictors
corr_matrix = np.corrcoef(Xk.values, rowvar=False)

# 2. Calculate Variance Inflation Factors (VIF)
# The VIFs are precisely the diagonal elements of the inverse of the correlation matrix
inverse_corr_matrix = np.linalg.inv(corr_matrix)
vifs = np.diagonal(inverse_corr_matrix)


print("======================================================================")
print(f"MULTICOLLINEARITY DIAGNOSTICS (VIF)")
print("======================================================================")
print(f"{'Predictor Variable':<25} | {'VIF Score':<10} | {'Status':<15}")
print("-" * 65)

for i in range(len(vifs)):
    vif_val = vifs[i]
    if vif_val > 5:
        status = "DANGER (>5)"
    elif vif_val > 2.5:
        status = "MODERATE"
    else:
        status = "SAFE"
    print(f"{feature_names[i]:<25} | {vif_val:<10.4f} | {status:<15}")

MULTICOLLINEARITY DIAGNOSTICS (VIF)
Predictor Variable        | VIF Score  | Status         
-----------------------------------------------------------------
Intercept                 | 1.6983     | SAFE           
Median Income             | 1.6691     | SAFE           
Vacancy Rate              | 1.4785     | SAFE           
Homeownership Rate        | 1.2385     | SAFE           


In [9]:
# MODEL TRAINING

X_train, X_test, Y_train, Y_test = train_test_split(Xk, Y, test_size=0.20, random_state=42)
r_squared_train, r2_adj_train, aic_train, betahat, se_betahat, p_values_array = mlr(X_train, Y_train)

betahat = np.atleast_1d(betahat)
p_values_array = np.atleast_1d(p_values_array)

# MODEL TESTING
n_test = len(Y_test)
X_test_intercept = np.column_stack((np.ones(n_test), X_test))

#Predictions based on test
Y_predictions = X_test_intercept @ betahat

# Calculate Accuracy
residuals = Y_test - Y_predictions
test_sse = np.sum(residuals**2)
test_sst = np.sum((Y_test - np.mean(Y_test))**2)

test_r_squared = 1 - (test_sse / test_sst)
#Average of what we were off by (in dollars)
test_rmse = np.sqrt(test_sse / n_test)

print("======================================================================")
print(f"MACHINE LEARNING EVALUATION: TRAIN VS. TEST")
print("======================================================================")
print(f"Data Split:           {len(Y_train)} Training Neighborhoods | {n_test} Testing Neighborhoods")
print(f"Training R-Squared:   {r_squared_train:.4f} (Accuracy on data it studied)")
print(f"Testing R-Squared:    {test_r_squared:.4f} (Accuracy on HIDDEN data)")
print(f"Test RMSE (Error):   ${test_rmse:.2f} (Average prediction error in dollars)")

print("\n======================================================================")
print(f"MODEL FIT SUMMARY (BASED ON TRAINING SET)")
print("======================================================================")
print(f"AIC Score:          {aic_train:.4f}")
print("======================================================================")
print(f"{'Predictor Variable':<22} | {'Coefficient':<12} | {'P-Value':<10} | {'Significant?':<12}")
print("-" * 65)

for i in range(len(feature_names)):
    sig = "YES" if p_values_array[i] < 0.05 else "NO"
    coef_str = f"{betahat[i]:.6f}"
    pval_str = f"{p_values_array[i]:.4f}"

    print(f"{feature_names[i]:<22} | {coef_str:<12} | {pval_str:<10} | {sig:<12}")
print("======================================================================")

MACHINE LEARNING EVALUATION: TRAIN VS. TEST
Data Split:           140 Training Neighborhoods | 35 Testing Neighborhoods
Training R-Squared:   0.8012 (Accuracy on data it studied)
Testing R-Squared:    0.8985 (Accuracy on HIDDEN data)
Test RMSE (Error):   $216.50 (Average prediction error in dollars)

MODEL FIT SUMMARY (BASED ON TRAINING SET)
AIC Score:          1978.3163
Predictor Variable     | Coefficient  | P-Value    | Significant?
-----------------------------------------------------------------
Intercept              | 850.602737   | 0.0000     | YES         
Median Income          | 0.011505     | 0.0000     | YES         
Vacancy Rate           | 1283.233337  | 0.0133     | YES         
Homeownership Rate     | -246.801868  | 0.0806     | NO          
New Construction Share | 895.987908   | 0.0142     | YES         


Cross Validation Test for Model 2

In [10]:

# ======================================================================
# CROSS-VALIDATION & FINAL MODEL TEST
# ======================================================================
# We split on df_model which you already cleaned in your block above
df_train, df_test = train_test_split(df_model, test_size=0.20, random_state=42)

Y_train = df_train['median_gross_rent'].values
Y_test = df_test['median_gross_rent'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n======================================================================")
print("Evaluating Model (Income, Vacancy, Homeownership, New Construction)...")
print("======================================================================")

# Update features to match your new Variable Selection
model_features = ['median_household_income', 'vacancy_rate', 'homeownership_rate', 'new_construction_share']

X_train_model = df_train[model_features].values
X_test_model = df_test[model_features].values

# 1. Run Cross-Validation on the Training Data
cv_r2_scores = []
cv_rmse_scores = []
fold_number = 1

for train_index, val_index in kf.split(X_train_model):
    X_fold_train, X_fold_val = X_train_model[train_index], X_train_model[val_index]
    Y_fold_train, Y_fold_val = Y_train[train_index], Y_train[val_index]

    # Train on 4 folds
    _, _, _, betahat_fold, _, _ = mlr(X_fold_train, Y_fold_train)

    # Validate on the 1 remaining fold
    n_val = len(Y_fold_val)
    X_fold_val_int = np.column_stack((np.ones(n_val), X_fold_val))
    fold_predictions = X_fold_val_int @ betahat_fold

    # Calculate Fold Metrics
    fold_sse = np.sum((Y_fold_val - fold_predictions)**2)
    fold_sst = np.sum((Y_fold_val - np.mean(Y_fold_val))**2)

    fold_r2 = 1 - (fold_sse / fold_sst)
    fold_rmse = np.sqrt(fold_sse / n_val)

    cv_r2_scores.append(fold_r2)
    cv_rmse_scores.append(fold_rmse)

    print(f"Fold {fold_number}: R-Squared = {fold_r2:.4f} | RMSE = ${fold_rmse:.2f}")
    fold_number += 1

# --- STANDARD DEVIATION ---
cv_r2_mean = np.mean(cv_r2_scores)
cv_r2_std = np.std(cv_r2_scores)

cv_rmse_mean = np.mean(cv_rmse_scores)
cv_rmse_std = np.std(cv_rmse_scores)

print("----------------------------------------------------------------------")
print(f"-> Cross-Validation R-Squared: {cv_r2_mean:.4f} (± {cv_r2_std:.4f})")
print(f"-> Cross-Validation RMSE:      ${cv_rmse_mean:.2f} (± ${cv_rmse_std:.2f})")
print("======================================================================")

# 2. Final Test!
_, _, _, betahat_final, _, _ = mlr(X_train_model, Y_train)

X_test_model_int = np.column_stack((np.ones(len(Y_test)), X_test_model))

final_predictions = X_test_model_int @ betahat_final
final_sse = np.sum((Y_test - final_predictions)**2)
final_sst = np.sum((Y_test - np.mean(Y_test))**2)
final_test_r2 = 1 - (final_sse / final_sst)
final_test_rmse = np.sqrt(final_sse / len(Y_test))

print(f"-> FINAL Test Set R-Squared:           {final_test_r2:.4f}")
print(f"-> FINAL Test Set RMSE:               ${final_test_rmse:.2f}\n")


Evaluating Model (Income, Vacancy, Homeownership, New Construction)...
Fold 1: R-Squared = 0.7730 | RMSE = $198.64
Fold 2: R-Squared = 0.6970 | RMSE = $287.93
Fold 3: R-Squared = 0.8014 | RMSE = $288.28
Fold 4: R-Squared = 0.6754 | RMSE = $391.33
Fold 5: R-Squared = 0.9016 | RMSE = $218.83
----------------------------------------------------------------------
-> Cross-Validation R-Squared: 0.7697 (± 0.0807)
-> Cross-Validation RMSE:      $277.00 (± $67.59)
-> FINAL Test Set R-Squared:           0.8985
-> FINAL Test Set RMSE:               $216.50



## Model Run #3

In [11]:
# VARIABLE SELECTION
cols_to_keep = ['median_gross_rent', 'median_household_income', 'stabilization_share', 'renter_density', 'pre_war_share']
df_model = df_merged[cols_to_keep].replace([np.inf, -np.inf], np.nan).dropna()

Y = df_model['median_gross_rent'].values
Xk = df_model[['median_household_income', 'stabilization_share', 'renter_density', 'pre_war_share']]

feature_names = ['Intercept', 'Median Income', 'Stabilization Share', 'Renter Density', 'Pre-War Share']

# Multicollinearity Check
# Calculate the Correlation Matrix of the predictors
corr_matrix = np.corrcoef(Xk.values, rowvar=False)

# Calculate Variance Inflation Factors
inverse_corr_matrix = np.linalg.inv(corr_matrix)
vifs = np.diagonal(inverse_corr_matrix)


print("======================================================================")
print(f"MULTICOLLINEARITY DIAGNOSTICS (VIF)")
print("======================================================================")
print(f"{'Predictor Variable':<25} | {'VIF Score':<10} | {'Status':<15}")
print("-" * 65)

for i in range(len(vifs)):
    vif_val = vifs[i]
    if vif_val > 5:
        status = "DANGER (>5)"
    elif vif_val > 2.5:
        status = "MODERATE"
    else:
        status = "SAFE"
    print(f"{feature_names[i]:<25} | {vif_val:<10.4f} | {status:<15}")


MULTICOLLINEARITY DIAGNOSTICS (VIF)
Predictor Variable        | VIF Score  | Status         
-----------------------------------------------------------------
Intercept                 | 1.1753     | SAFE           
Median Income             | 1.5528     | SAFE           
Stabilization Share       | 1.7733     | SAFE           
Renter Density            | 1.0731     | SAFE           


In [12]:
# MODEL TRAINING

X_train, X_test, Y_train, Y_test = train_test_split(Xk, Y, test_size=0.20, random_state=42)
r_squared_train, r2_adj_train, aic_train, betahat, se_betahat, p_values_array = mlr(X_train, Y_train)

betahat = np.atleast_1d(betahat)
p_values_array = np.atleast_1d(p_values_array)

# MODEL TESTING
n_test = len(Y_test)
X_test_intercept = np.column_stack((np.ones(n_test), X_test))

#Predictions based on test
Y_predictions = X_test_intercept @ betahat

# Calculate Accuracy
residuals = Y_test - Y_predictions
test_sse = np.sum(residuals**2)
test_sst = np.sum((Y_test - np.mean(Y_test))**2)

test_r_squared = 1 - (test_sse / test_sst)
#Average of what we were off by (in dollars)
test_rmse = np.sqrt(test_sse / n_test)

print("======================================================================")
print(f"MACHINE LEARNING EVALUATION: TRAIN VS. TEST")
print("======================================================================")
print(f"Data Split:           {len(Y_train)} Training Neighborhoods | {n_test} Testing Neighborhoods")
print(f"Training R-Squared:   {r_squared_train:.4f} (Accuracy on data it studied)")
print(f"Testing R-Squared:    {test_r_squared:.4f} (Accuracy on HIDDEN data)")
print(f"Test RMSE (Error):   ${test_rmse:.2f} (Average prediction error in dollars)")

print("\n======================================================================")
print(f"MODEL FIT SUMMARY (BASED ON TRAINING SET)")
print("======================================================================")
print(f"AIC Score:          {aic_train:.4f}")
print("======================================================================")
print(f"{'Predictor Variable':<22} | {'Coefficient':<12} | {'P-Value':<10} | {'Significant?':<12}")
print("-" * 65)

for i in range(len(feature_names)):
    sig = "YES" if p_values_array[i] < 0.05 else "NO"
    coef_str = f"{betahat[i]:.6f}"
    pval_str = f"{p_values_array[i]:.4f}"

    print(f"{feature_names[i]:<22} | {coef_str:<12} | {pval_str:<10} | {sig:<12}")
print("======================================================================")

MACHINE LEARNING EVALUATION: TRAIN VS. TEST
Data Split:           140 Training Neighborhoods | 35 Testing Neighborhoods
Training R-Squared:   0.7776 (Accuracy on data it studied)
Testing R-Squared:    0.8698 (Accuracy on HIDDEN data)
Test RMSE (Error):   $245.20 (Average prediction error in dollars)

MODEL FIT SUMMARY (BASED ON TRAINING SET)
AIC Score:          1994.0153
Predictor Variable     | Coefficient  | P-Value    | Significant?
-----------------------------------------------------------------
Intercept              | 536.367881   | 0.0000     | YES         
Median Income          | 0.013175     | 0.0000     | YES         
Stabilization Share    | -63.493498   | 0.6192     | NO          
Renter Density         | 544.035993   | 0.0016     | YES         
Pre-War Share          | -129.644368  | 0.3652     | NO          


Cross Validation for Model #3

In [13]:
# ======================================================================
# CROSS-VALIDATION & FINAL MODEL TEST
# ======================================================================
# We split on df_model which you already cleaned in your block above
df_train, df_test = train_test_split(df_model, test_size=0.20, random_state=42)

Y_train = df_train['median_gross_rent'].values
Y_test = df_test['median_gross_rent'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n======================================================================")
print("Evaluating Model (Income, Stabilization, Renter Density, Pre-War)...")
print("======================================================================")

# Update features to match your new Variable Selection
model_features = ['median_household_income', 'stabilization_share', 'renter_density', 'pre_war_share']

X_train_model = df_train[model_features].values
X_test_model = df_test[model_features].values

# 1. Run Cross-Validation on the Training Data
cv_r2_scores = []
cv_rmse_scores = []
fold_number = 1

for train_index, val_index in kf.split(X_train_model):
    X_fold_train, X_fold_val = X_train_model[train_index], X_train_model[val_index]
    Y_fold_train, Y_fold_val = Y_train[train_index], Y_train[val_index]

    # Train on 4 folds
    _, _, _, betahat_fold, _, _ = mlr(X_fold_train, Y_fold_train)

    # Validate on the 1 remaining fold
    n_val = len(Y_fold_val)
    X_fold_val_int = np.column_stack((np.ones(n_val), X_fold_val))
    fold_predictions = X_fold_val_int @ betahat_fold

    # Calculate Fold Metrics
    fold_sse = np.sum((Y_fold_val - fold_predictions)**2)
    fold_sst = np.sum((Y_fold_val - np.mean(Y_fold_val))**2)

    fold_r2 = 1 - (fold_sse / fold_sst)
    fold_rmse = np.sqrt(fold_sse / n_val)

    cv_r2_scores.append(fold_r2)
    cv_rmse_scores.append(fold_rmse)

    print(f"Fold {fold_number}: R-Squared = {fold_r2:.4f} | RMSE = ${fold_rmse:.2f}")
    fold_number += 1

# ---


Evaluating Model (Income, Stabilization, Renter Density, Pre-War)...
Fold 1: R-Squared = 0.7978 | RMSE = $187.47
Fold 2: R-Squared = 0.6669 | RMSE = $301.87
Fold 3: R-Squared = 0.7481 | RMSE = $324.62
Fold 4: R-Squared = 0.6117 | RMSE = $428.04
Fold 5: R-Squared = 0.8777 | RMSE = $243.95


## Model Run #4

In [14]:
#VARIABLE SELECTION
cols_to_keep = [
    'median_gross_rent', 'median_household_income', 'stabilization_share',
    'vacancy_rate', 'homeownership_rate', 'new_construction_share', 'pre_war_share'
]
df_model = df_merged[cols_to_keep].replace([np.inf, -np.inf], np.nan).dropna()

Y = df_model['median_gross_rent'].values
Xk = df_model[[
    'median_household_income', 'stabilization_share', 'vacancy_rate',
    'homeownership_rate', 'new_construction_share', 'pre_war_share'
]]

feature_names = [
    'Intercept', 'Median Income', 'Stabilization Share', 'Vacancy Rate',
    'Homeownership Rate', 'New Construction Share', 'Pre-War Share'
]

# Multicollinearity Check
# Calculate the Correlation Matrix of the predictors
corr_matrix = np.corrcoef(Xk.values, rowvar=False)

# Calculate Variance Inflation Factors (VIF)
inverse_corr_matrix = np.linalg.inv(corr_matrix)
vifs = np.diagonal(inverse_corr_matrix)


print("======================================================================")
print(f"MULTICOLLINEARITY DIAGNOSTICS (VIF)")
print("======================================================================")
print(f"{'Predictor Variable':<25} | {'VIF Score':<10} | {'Status':<15}")
print("-" * 65)

for i in range(len(vifs)):
    vif_val = vifs[i]
    if vif_val > 5:
        status = "DANGER (>5)"
    elif vif_val > 2.5:
        status = "MODERATE"
    else:
        status = "SAFE"
    print(f"{feature_names[i]:<25} | {vif_val:<10.4f} | {status:<15}")

MULTICOLLINEARITY DIAGNOSTICS (VIF)
Predictor Variable        | VIF Score  | Status         
-----------------------------------------------------------------
Intercept                 | 1.7696     | SAFE           
Median Income             | 1.5593     | SAFE           
Stabilization Share       | 1.7631     | SAFE           
Vacancy Rate              | 2.3214     | SAFE           
Homeownership Rate        | 1.4038     | SAFE           
New Construction Share    | 1.2207     | SAFE           


In [15]:
# MODEL TRAINING

X_train, X_test, Y_train, Y_test = train_test_split(Xk, Y, test_size=0.20, random_state=42)
r_squared_train, r2_adj_train, aic_train, betahat, se_betahat, p_values_array = mlr(X_train, Y_train)

betahat = np.atleast_1d(betahat)
p_values_array = np.atleast_1d(p_values_array)

# MODEL TESTING
n_test = len(Y_test)
X_test_intercept = np.column_stack((np.ones(n_test), X_test))

#Predictions based on test
Y_predictions = X_test_intercept @ betahat

# Calculate Accuracy
residuals = Y_test - Y_predictions
test_sse = np.sum(residuals**2)
test_sst = np.sum((Y_test - np.mean(Y_test))**2)

test_r_squared = 1 - (test_sse / test_sst)
#Average of what we were off by (in dollars)
test_rmse = np.sqrt(test_sse / n_test)

print("======================================================================")
print(f"MACHINE LEARNING EVALUATION: TRAIN VS. TEST")
print("======================================================================")
print(f"Data Split:           {len(Y_train)} Training Neighborhoods | {n_test} Testing Neighborhoods")
print(f"Training R-Squared:   {r_squared_train:.4f} (Accuracy on data it studied)")
print(f"Testing R-Squared:    {test_r_squared:.4f} (Accuracy on HIDDEN data)")
print(f"Test RMSE (Error):   ${test_rmse:.2f} (Average prediction error in dollars)")

print("\n======================================================================")
print(f"MODEL FIT SUMMARY (BASED ON TRAINING SET)")
print("======================================================================")
print(f"AIC Score:          {aic_train:.4f}")
print("======================================================================")
print(f"{'Predictor Variable':<22} | {'Coefficient':<12} | {'P-Value':<10} | {'Significant?':<12}")
print("-" * 65)

for i in range(len(feature_names)):
    sig = "YES" if p_values_array[i] < 0.05 else "NO"
    coef_str = f"{betahat[i]:.6f}"
    pval_str = f"{p_values_array[i]:.4f}"

    print(f"{feature_names[i]:<22} | {coef_str:<12} | {pval_str:<10} | {sig:<12}")
print("======================================================================")

MACHINE LEARNING EVALUATION: TRAIN VS. TEST
Data Split:           140 Training Neighborhoods | 35 Testing Neighborhoods
Training R-Squared:   0.8015 (Accuracy on data it studied)
Testing R-Squared:    0.9001 (Accuracy on HIDDEN data)
Test RMSE (Error):   $214.69 (Average prediction error in dollars)

MODEL FIT SUMMARY (BASED ON TRAINING SET)
AIC Score:          1982.1092
Predictor Variable     | Coefficient  | P-Value    | Significant?
-----------------------------------------------------------------
Intercept              | 890.635485   | 0.0000     | YES         
Median Income          | 0.011591     | 0.0000     | YES         
Stabilization Share    | -38.784397   | 0.7501     | NO          
Vacancy Rate           | 1240.660017  | 0.0202     | YES         
Homeownership Rate     | -295.303102  | 0.1042     | NO          
New Construction Share | 849.074873   | 0.0307     | YES         
Pre-War Share          | -44.295977   | 0.7613     | NO          


Cross Validation for Model #4

In [16]:
# ======================================================================
# CROSS-VALIDATION & FINAL MODEL TEST
# ======================================================================
# We split on df_model which you already cleaned in your block above
df_train, df_test = train_test_split(df_model, test_size=0.20, random_state=42)

Y_train = df_train['median_gross_rent'].values
Y_test = df_test['median_gross_rent'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n======================================================================")
print("Evaluating Model (The Master Mix: 6 Features)...")
print("======================================================================")

# Update features to match your new Variable Selection
model_features = [
    'median_household_income', 'stabilization_share', 'vacancy_rate',
    'homeownership_rate', 'new_construction_share', 'pre_war_share'
]

X_train_model = df_train[model_features].values
X_test_model = df_test[model_features].values

# 1. Run Cross-Validation on the Training Data
cv_r2_scores = []
cv_rmse_scores = []
fold_number = 1

for train_index, val_index in kf.split(X_train_model):
    X_fold_train, X_fold_val = X_train_model[train_index], X_train_model[val_index]
    Y_fold_train, Y_fold_val = Y_train[train_index], Y_train[val_index]

    # Train on 4 folds
    _, _, _, betahat_fold, _, _ = mlr(X_fold_train, Y_fold_train)

    # Validate on the 1 remaining fold
    n_val = len(Y_fold_val)
    X_fold_val_int = np.column_stack((np.ones(n_val), X_fold_val))
    fold_predictions = X_fold_val_int @ betahat_fold

    # Calculate Fold Metrics
    fold_sse = np.sum((Y_fold_val - fold_predictions)**2)
    fold_sst = np.sum((Y_fold_val - np.mean(Y_fold_val))**2)

    fold_r2 = 1 - (fold_sse / fold_sst)
    fold_rmse = np.sqrt(fold_sse / n_val)

    cv_r2_scores.append(fold_r2)
    cv_rmse_scores.append(fold_rmse)

    print(f"Fold {fold_number}: R-Squared = {fold_r2:.4f} | RMSE = ${fold_rmse:.2f}")
    fold_number += 1

# --- STANDARD DEVIATION ---
cv_r2_mean = np.mean(cv_r2_scores)
cv_r2_std = np.std(cv_r2_scores)

cv_rmse_mean = np.mean(cv_rmse_scores)
cv_rmse_std = np.std(cv_rmse_scores)

print("----------------------------------------------------------------------")
print(f"-> Cross-Validation R-Squared: {cv_r2_mean:.4f} (± {cv_r2_std:.4f})")
print(f"-> Cross-Validation RMSE:      ${cv_rmse_mean:.2f} (± ${cv_rmse_std:.2f})")
print("======================================================================")

# 2. Final Test!
_, _, _, betahat_final, _, _ = mlr(X_train_model, Y_train)

X_test_model_int = np.column_stack((np.ones(len(Y_test)), X_test_model))

final_predictions = X_test_model_int @ betahat_final
final_sse = np.sum((Y_test - final_predictions)**2)
final_sst = np.sum((Y_test - np.mean(Y_test))**2)
final_test_r2 = 1 - (final_sse / final_sst)
final_test_rmse = np.sqrt(final_sse / len(Y_test))

print(f"-> FINAL Test Set R-Squared:           {final_test_r2:.4f}")
print(f"-> FINAL Test Set RMSE:               ${final_test_rmse:.2f}\n")


Evaluating Model (The Master Mix: 6 Features)...
Fold 1: R-Squared = 0.7488 | RMSE = $208.95
Fold 2: R-Squared = 0.6980 | RMSE = $287.42
Fold 3: R-Squared = 0.7670 | RMSE = $312.25
Fold 4: R-Squared = 0.6477 | RMSE = $407.73
Fold 5: R-Squared = 0.9005 | RMSE = $220.04
----------------------------------------------------------------------
-> Cross-Validation R-Squared: 0.7524 (± 0.0850)
-> Cross-Validation RMSE:      $287.28 (± $71.82)
-> FINAL Test Set R-Squared:           0.9001
-> FINAL Test Set RMSE:               $214.69

